In [ ]:
%pip install -q pyannote.audio faster-whisper

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import torch
from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio
from pyannote.audio import Pipeline

# ---------- تنظیمات ----------
AUDIO_PATH = Path("/home/ali/Downloads/synthetic_persian_two_speakers (1).wav")
OUTPUT_PATH = Path("output.json")
ASR_MODEL_ID = "AmirMohseni/whisper-large-v3-persian-ct2-int8"
DIARIZATION_MODEL_ID = "pyannote/speaker-diarization-community-1"
NUM_SPEAKERS = 2  # برای فایل نمونه دو گوینده داریم؛ اگر واقعاً نامشخص است None بگذارید
MERGE_SAME_SPEAKER_GAP = 0.35  # ادغام turnهای پشت‌سرهم یک گوینده
ASR_PADDING_SECONDS = 0.15      # جلوگیری از بریده‌شدن ابتدا/انتهای کلمات

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"فایل صوتی پیدا نشد: {AUDIO_PATH.resolve()}")

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: " ).strip()
if not HF_TOKEN:
    raise ValueError("برای دانلود مدل pyannote باید HF_TOKEN وارد شود.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "int8_float16" if DEVICE == "cuda" else "int8"
print({"audio": str(AUDIO_PATH), "device": DEVICE, "compute_type": COMPUTE_TYPE})

## ۱) استخراج timestamp گوینده‌ها با pyannote

In [ ]:
diarization_pipeline = Pipeline.from_pretrained(
    DIARIZATION_MODEL_ID,
    token=HF_TOKEN,
)
diarization_pipeline.to(torch.device(DEVICE))

diarization_options = {}
if NUM_SPEAKERS is not None:
    diarization_options["num_speakers"] = NUM_SPEAKERS

diarization_output = diarization_pipeline(
    str(AUDIO_PATH),
    **diarization_options,
)

# exclusive diarization باعث می‌شود در هر لحظه فقط یک گوینده داشته باشیم.
diarization = getattr(diarization_output, "exclusive_speaker_diarization", None)
if diarization is None:
    diarization = diarization_output.speaker_diarization

raw_speaker_turns = sorted(
    [
        {
            "speaker": str(speaker),
            "start": float(turn.start),
            "end": float(turn.end),
        }
        for turn, speaker in diarization
    ],
    key=lambda item: item["start"],
)

# قطعه‌های مجاور یک گوینده که gap بسیار کوتاه دارند یکی می‌شوند.
speaker_turns = []
for turn in raw_speaker_turns:
    if (
        speaker_turns
        and speaker_turns[-1]["speaker"] == turn["speaker"]
        and turn["start"] - speaker_turns[-1]["end"] <= MERGE_SAME_SPEAKER_GAP
    ):
        speaker_turns[-1]["end"] = max(speaker_turns[-1]["end"], turn["end"])
    else:
        speaker_turns.append(turn.copy())

print(f"{len(raw_speaker_turns)} raw turns -> {len(speaker_turns)} merged turns")
speaker_turns

## ۲) بارگذاری مدل ASR و صوت

از word timestamp مدل استفاده نمی‌کنیم؛ چون alignment این مدل برای فایل نمونه کلمات را به بازه‌های اشتباه منتقل کرده است.

In [ ]:
asr_model = WhisperModel(
    ASR_MODEL_ID,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

audio = decode_audio(str(AUDIO_PATH), sampling_rate=16_000)
audio_duration = len(audio) / 16_000
print(f"audio loaded: {audio_duration:.2f}s")

## ۳) برش هر speaker turn، تبدیل مستقل به متن، و ساخت JSON

هر بازهٔ pyannote مستقیماً از waveform بریده و جداگانه transcribe می‌شود. بنابراین خرابی word alignment نمی‌تواند متن را به turn دیگری منتقل کند و `start`/`end` خروجی دقیقاً از pyannote می‌آید.

In [ ]:
if not speaker_turns:
    raise RuntimeError("pyannote هیچ speaker turnی پیدا نکرد.")

result = []
for index, turn in enumerate(speaker_turns):
    # padding کوتاه فقط برای جلوگیری از بریده‌شدن فونم‌های مرزی است.
    clip_start = max(0.0, turn["start"] - ASR_PADDING_SECONDS)
    clip_end = min(audio_duration, turn["end"] + ASR_PADDING_SECONDS)
    start_sample = round(clip_start * 16_000)
    end_sample = round(clip_end * 16_000)
    clip = audio[start_sample:end_sample]

    segments_iterator, _ = asr_model.transcribe(
        clip,
        language="fa",
        task="transcribe",
        beam_size=5,
        temperature=0.0,
        word_timestamps=False,
        condition_on_previous_text=False,
        vad_filter=False,
    )
    text = " ".join(
        segment.text.strip() for segment in segments_iterator if segment.text.strip()
    ).strip()
    if not text:
        continue
    result.append({
        "speaker": turn["speaker"],
        "start": round(turn["start"], 3),
        "end": round(turn["end"], 3),
        "text": text,
    })
    print(f"[{index + 1}/{len(speaker_turns)}] {turn['speaker']} "
          f"{turn['start']:.2f}-{turn['end']:.2f}: {text}")

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(json.dumps(result, ensure_ascii=False, indent=2))
print(f"\nSaved to: {OUTPUT_PATH.resolve()}")